# Conteo de Personas con Wi-Fi CSI
## Device-Free Sensing con ESP32 + Random Forest

**Descripción:** Entrena modelos Random Forest para predecir el número de personas
(0-7) en una habitación usando datos CSI capturados con ESP32.

### Tres niveles de clasificación:
1. **Binario**: Vacío (0) vs Ocupado (1-7)
2. **3 clases**: Vacío (0) / Pocos (1-3) / Muchos (4-7)
3. **8 clases**: 0, 1, 2, 3, 4, 5, 6, 7 personas

In [ ]:
import os, re, glob, csv, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score, f1_score)
from sklearn.preprocessing import StandardScaler
import joblib
from collections import Counter

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('OK')

In [ ]:
DATA_DIR = 'datos'
WINDOW_SIZE = 20
OVERLAP = 0.5
CSI_START_IDX = 12

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, 'csi_*.csv')))
print(f'Archivos: {len(csv_files)}')

In [ ]:
def parse_csi(s):
    return np.array([int(x) for x in re.findall(r'-?\d+', s)], dtype=np.int16)

def iq_to_amp(arr, start=CSI_START_IDX):
    iq = arr[start:]
    n = len(iq) // 2
    return np.sqrt(iq[0::2].astype(np.float32)**2 + iq[1::2].astype(np.float32)**2)

def load_csi(filepath):
    amps, metas, label = [], [], None
    with open(filepath) as f:
        reader = csv.reader(f)
        h = next(reader)
        ci, li, rssi_i, nf_i = h.index('CSI_DATA'), h.index('n_personas'), h.index('rssi'), h.index('noise_floor')
        for row in reader:
            if len(row) != 27:
                continue
            raw = row[ci]
            raw = re.sub(r'\[([^\]]*)\]', lambda m: '[' + m.group(1).replace(',', ' ') + ']', raw)
            amps.append(iq_to_amp(parse_csi(raw)))
            metas.append([int(row[rssi_i]), int(row[nf_i])])
            if label is None:
                label = int(row[li])
    return np.array(amps, dtype=np.float32), np.array(metas, dtype=np.int16), label

In [ ]:
def segment_windows(amps, ws=WINDOW_SIZE, ol=OVERLAP):
    step = max(1, int(ws * (1 - ol)))
    return np.array([amps[s:s+ws] for s in range(0, len(amps)-ws+1, step)])

def extract_features(w, meta_w):
    feats = []
    for sc in range(w.shape[1]):
        c = w[:, sc]
        feats += [c.mean(), c.std(), np.var(c), np.percentile(c,25), np.percentile(c,75), np.ptp(c)]
    feats += [w.mean(), w.std(), np.mean(np.std(w,axis=0)), np.std(np.mean(w,axis=0))]
    feats += [meta_w[:,0].mean(), meta_w[:,0].std(), meta_w[:,1].mean(), meta_w[:,1].std()]
    return np.array(feats)

In [ ]:
amps_p0, meta_p0, lbl = load_csi(csv_files[0])
print(f'Ejemplo: {os.path.basename(csv_files[0])}: {lbl} pers')
print(f'Amplitudes: {amps_p0.shape}, RSSI medio: {meta_p0[:,0].mean():.0f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].plot(amps_p0[:200, 20], alpha=.7)
axes[0].set(title='Subc 20 - Vacío', xlabel='Paquete', ylabel='Amplitud')
axes[1].imshow(amps_p0[:100].T, aspect='auto', cmap='viridis')
axes[1].set(title='Heatmap CSI vacío', xlabel='Paquete', ylabel='Subc')
axes[2].plot(meta_p0[:, 0], alpha=.7, label='RSSI')
axes[2].set(title='RSSI', xlabel='Paquete', ylabel='dBm')
plt.tight_layout(); plt.show()

In [ ]:
def build_dataset(csv_files):
    X, y = [], []
    for f in csv_files:
        amps, meta, lbl = load_csi(f)
        windows = segment_windows(amps)
        for i, w in enumerate(windows):
            s = i * max(1, int(WINDOW_SIZE * (1 - OVERLAP)))
            mw = meta[s:s+WINDOW_SIZE]
            if len(mw) != WINDOW_SIZE:
                continue
            X.append(extract_features(w, mw))
            y.append(lbl)
    return np.array(X), np.array(y)

X, y = build_dataset(csv_files)
print(f'X: {X.shape}, y: {y.shape}')
for c in range(8):
    print(f'  {c} pers: {(y==c).sum()}')

In [ ]:
# Dividir por sesión
X_train, X_test, y_train, y_test = [], [], [], []
for f in csv_files:
    amps, meta, lbl = load_csi(f)
    windows = segment_windows(amps)
    feats, lbls_list = [], []
    for i, w in enumerate(windows):
        s = i * max(1, int(WINDOW_SIZE * (1 - OVERLAP)))
        mw = meta[s:s+WINDOW_SIZE]
        if len(mw) != WINDOW_SIZE:
            continue
        feats.append(extract_features(w, mw))
        lbls_list.append(lbl)
    feats = np.array(feats)
    lbls_arr = np.array(lbls_list)
    (X_train if '_s1_' in f else X_test).append(feats)
    (y_train if '_s1_' in f else y_test).append(lbls_arr)

X_train = np.vstack(X_train); y_train = np.concatenate(y_train)
X_test  = np.vstack(X_test);  y_test  = np.concatenate(y_test)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')

---
## 1. Clasificación Binaria: Vacío vs Ocupado

In [ ]:
y2_train = (y_train > 0).astype(int)
y2_test  = (y_test > 0).astype(int)

rf2 = RandomForestClassifier(200, max_depth=15, random_state=42, n_jobs=-1)
rf2.fit(X_train_s, y2_train)
p2 = rf2.predict(X_test_s)

print(f'Accuracy: {accuracy_score(y2_test, p2):.4f}')
print(f'F1:       {f1_score(y2_test, p2):.4f}')
print()
print(classification_report(y2_test, p2, target_names=['Vacío', 'Ocupado'], digits=4))

cm = confusion_matrix(y2_test, p2)
ConfusionMatrixDisplay(cm, display_labels=['Vacío','Ocupado']).plot(values_format='d', cmap='Blues')
plt.title('Matriz de Confusión - Binario'); plt.show()

cv = cross_val_score(rf2, X_train_s, y2_train, cv=5)
print(f'CV: {cv.mean():.4f} +/- {cv.std():.4f}')

---
## 2. Clasificación en 3 categorías: Vacío (0) / Bajo (1-3) / Alto (4-7)

In [ ]:
def to_3class(y):
    return np.where(y == 0, 0, np.where(y <= 3, 1, 2))

y3_train = to_3class(y_train)
y3_test  = to_3class(y_test)

rf3 = RandomForestClassifier(200, max_depth=18, random_state=42, n_jobs=-1)
rf3.fit(X_train_s, y3_train)
p3 = rf3.predict(X_test_s)

print(f'Accuracy: {accuracy_score(y3_test, p3):.4f}')
print(f'F1-macro: {f1_score(y3_test, p3, average="macro"):.4f}')
print()
print(classification_report(y3_test, p3, target_names=['Vacío','1-3','4-7'], digits=4))

cm = confusion_matrix(y3_test, p3)
ConfusionMatrixDisplay(cm, display_labels=['Vacío','1-3','4-7']).plot(values_format='d', cmap='Blues')
plt.title('Matriz de Confusión - 3 clases'); plt.show()

---
## 3. Clasificación en 8 clases (0-7 personas)

In [ ]:
rf8 = RandomForestClassifier(300, max_depth=22, random_state=42, n_jobs=-1)
rf8.fit(X_train_s, y_train)
p8 = rf8.predict(X_test_s)

acc = accuracy_score(y_test, p8)
f1 = f1_score(y_test, p8, average='macro')
print(f'Accuracy: {acc:.4f}  F1-macro: {f1:.4f}')
print()
print(classification_report(y_test, p8, digits=4))

cm = confusion_matrix(y_test, p8)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(cm, display_labels=range(8)).plot(ax=ax[0], values_format='d', cmap='Blues')
ax[0].set_title('Confusión (conteos)')
cm_n = cm.astype(float) / cm.sum(axis=1)[:, None]
ConfusionMatrixDisplay(cm_n, display_labels=range(8)).plot(ax=ax[1], values_format='.2f', cmap='Blues')
ax[1].set_title('Confusión (normalizada)')
plt.tight_layout(); plt.show()

cv = cross_val_score(rf8, X_train_s, y_train, cv=5)
print(f'CV 5-fold: {cv}  media={cv.mean():.4f} std={cv.std():.4f}')

In [ ]:
# Top features
top = 20
imp = rf8.feature_importances_
idx = np.argsort(imp)[::-1][:top]
plt.figure(figsize=(10,4))
plt.bar(range(top), imp[idx])
plt.xticks(range(top), [f'F{i}' for i in idx], rotation=45)
plt.title(f'Top {top} Características - RF 8 clases')
plt.tight_layout(); plt.show()

In [ ]:
# Visualización: Heatmaps por clase
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for c in range(8):
    a, _, _ = load_csi(csv_files[c*2])  # sesión 1
    im = axes.flat[c].imshow(a[:100].T, aspect='auto', cmap='viridis')
    axes.flat[c].set(title=f'{c} pers')
    plt.colorbar(im, ax=axes.flat[c], shrink=.6)
plt.suptitle('Amplitud CSI por clase (100 paquetes, 64 subportadoras)')
plt.tight_layout(); plt.show()

In [ ]:
# Guardar modelos
os.makedirs('modelo', exist_ok=True)
joblib.dump(rf2, 'modelo/rf_binario.pkl')
joblib.dump(rf3, 'modelo/rf_3clases.pkl')
joblib.dump(rf8, 'modelo/rf_8clases.pkl')
joblib.dump(scaler, 'modelo/scaler.pkl')
print('Modelos guardados en modelo/')

---
## Conclusiones

| Clasificación | Accuracy | F1-macro | Uso práctico |
|---|---|---|---|
| Binario (vacío/ocupado) | Alta | Alta | Detección de presencia |
| 3 clases (0/1-3/4-7) | Media | Media | Control HVAC, iluminación |
| 8 clases (0-7) | Baja | Baja | Conteo preciso (difícil) |

El modelo binario es el más robusto para aplicaciones prácticas.
Para mejorar las 8 clases se necesita más data, features en dominio de frecuencia,
y técnicas de deep learning (CNN/ResNet como en los papers de referencia).